In [1]:
import numpy as np
import pandas as pd
import string

from datetime import datetime, timedelta


# data import

## node zone

In [2]:
dtype_map = {
    "agg_pjm_ExternalNodeID": "string",
    "pjm_ExternalNodeID": "string",
    "agg_NodeName": "string",
    "NodeName": "string",
    "forecast_area": "string",
    "is_valid": "int8"
}

offset = 0
today_str = (datetime.today() - timedelta(days=offset)).strftime("%m%d") 

hourly_load = pd.read_csv(
    f"data/hourly_load_MW_by_distribution/hourly_load_MW_by_distribution_{today_str}.csv",
    skiprows=[1],
    dtype=dtype_map,
    parse_dates=["datetime_ending_ept", "insert_datetime"]
)

hourly_load.head()

,autokey,datetime_ending_ept,agg_pjm_ExternalNodeID,agg_NodeName,pjm_ExternalNodeID,NodeName,distribution_factor,forecast_area,forecast_load_mw,distributed_load_mw,is_valid,insert_datetime
0,100831683,2026-05-16 01:00:00,8445784,AEP,32412197,ORDNANCE138 KV T2,0.00017,AEP,13704.0,2.31598,1,2026-05-16 08:39:33
1,100831684,2026-05-16 01:00:00,8445784,AEP,32412199,ORDNANCE138 KV T3,0.00017,AEP,13704.0,2.31598,1,2026-05-16 08:39:33
2,100831685,2026-05-16 01:00:00,8445784,AEP,32412201,OREBANK 138 KV T1,0.00021,AEP,13704.0,2.82302,1,2026-05-16 08:39:33
3,100831686,2026-05-16 01:00:00,8445784,AEP,32412203,OREBANK 138 KV T2,0.00021,AEP,13704.0,2.82302,1,2026-05-16 08:39:33
4,100831687,2026-05-16 01:00:00,8445784,AEP,2156109901,ORONOKO 69 KV LOAD1,0.00008,AEP,13704.0,1.12373,1,2026-05-16 08:39:33


In [3]:
hourly_load['agg_NodeName'].unique()

<ArrowStringArray>
[    'AEP',    'ATSI',     'DOM',   'COMED',     'APS',     'PPL',    'PSEG',
    'DEOK',    'EKPC',    'PECO',     'DAY',     'DPL', 'PENELEC',     'BGE',
   'METED',   'PEPCO',    'JCPL',    'AECO',     'DUQ',    'RECO']
Length: 20, dtype: string

In [4]:
hourly_load.sort_values("datetime_ending_ept", inplace=True)

hourly_load = hourly_load[hourly_load["datetime_ending_ept"] >= "2026-03-12"].copy()

In [5]:
hourly_load.columns = (
    hourly_load.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

hourly_load = hourly_load.drop(columns=[
    "autokey",
    "insert_datetime"
], errors="ignore")

hourly_load["distribution_factor"] = (
    hourly_load["distribution_factor"]
    .astype(float)
    .round(8)
)

In [6]:
hourly_load = hourly_load.dropna(subset=[
    "datetime_ending_ept",
    "nodename",
    "distributed_load_mw"
])

hourly_load = hourly_load.drop_duplicates(
    subset=["datetime_ending_ept", "nodename"]
)

hourly_load.head()

,datetime_ending_ept,agg_pjm_externalnodeid,agg_nodename,pjm_externalnodeid,nodename,distribution_factor,forecast_area,forecast_load_mw,distributed_load_mw,is_valid
9494387,2026-04-01,34508503,DAY,40243459,DAYTONMA69 KV BK-3,0.00465,DAYTON,1816.0,8.43714,1
1327721,2026-04-01,51297,PECO,48955,MACDADE 13 KV 1BUS,0.00379,PECO/MIDATL,3620.0,13.72704,1
1327722,2026-04-01,51297,PECO,48949,LOMBARD 13 KV NBU9,0.00225,PECO/MIDATL,3620.0,8.14138,1
1327723,2026-04-01,51297,PECO,659720,LOMBARD 13 KV NBU4,0.00162,PECO/MIDATL,3620.0,5.86802,1
1327724,2026-04-01,51297,PECO,48948,LOMBARD 13 KV NBU3,0.00167,PECO/MIDATL,3620.0,6.05988,1


In [7]:
hourly_load = hourly_load[["datetime_ending_ept", "agg_nodename", "pjm_externalnodeid", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].copy()

hourly_load.loc[
    hourly_load["nodename"] == "SHAWVILL34 KV   3TX_LD",
    "nodename"
] = "SHAWVILL34 KV   CLR2"

hourly_load = (
    hourly_load.groupby(["datetime_ending_ept", "nodename"], as_index=False)
      .agg({
          "agg_nodename": "first",
          "pjm_externalnodeid": "first",
          "distribution_factor": "sum",
          "forecast_area": "first",
          "forecast_load_mw": "first",
          "distributed_load_mw": "sum"
      })
)

hourly_load = hourly_load.rename(columns={
    "pjm_externalnodeid": "externalnodeid"
})

hourly_load["clean_nodename"] = (
    hourly_load["nodename"]
    .str.replace(r"[ _]", "", regex=True)
    .str.upper()
)

hourly_load.head()

,datetime_ending_ept,nodename,agg_nodename,externalnodeid,distribution_factor,forecast_area,forecast_load_mw,distributed_load_mw,clean_nodename
0,2026-04-01,02ANGOLA138 KV TR1,ATSI,93353477,0.00078,ATSI,6931.0,5.39232,02ANGOLA138KVTR1
1,2026-04-01,02ANGOLA138 KV TR2,ATSI,93353479,0.00056,ATSI,6931.0,3.85364,02ANGOLA138KVTR2
2,2026-04-01,02ARMCO 138 KV TR1,ATSI,98369977,0.00128,ATSI,6931.0,8.85782,02ARMCO138KVTR1
3,2026-04-01,02ARMCO 138 KV TR2,ATSI,98369979,0.00128,ATSI,6931.0,8.85782,02ARMCO138KVTR2
4,2026-04-01,02ARMCO 138 KV TR3,ATSI,98369981,0.00128,ATSI,6931.0,8.85782,02ARMCO138KVTR3


## node bus

In [8]:
dtype_map = {
    "psse_bus_#": "string",
    "substation": "string",
    "nodename": "string",
    "nodekey": "string",
    "externalnodeid": "string",
    "lon": "float64",
    "lat": "float64",
}

pnode_to_bus = pd.read_csv('data/pnode_with_loc.csv')
pnode_to_bus.columns = (
    pnode_to_bus.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

pnode_to_bus = pnode_to_bus.astype({
    "externalnodeid": "Int64",
    "nodekey": "Int64"  # optional, if numeric
})

pnode_to_bus = pnode_to_bus.astype(dtype_map)

############################################################
nodename_mapping = {
    "BENJAMIN138 KV  BENDATLD": "BENJAMIN138 KV  DATCENLD",
    "960 ELGI4 KV    LAUXBLUE": "960 ELGI4 KV    BLUAUXLD",
    "960 ELGI4 KV    LAUXRED": "960 ELGI4 KV    REDAUXLD",
    "13 CRAWF138 KV  ATR57R04": "13 CRAWF138 KV  TR57R04",
    "13 CRAWF138 KV  ATR58R04": "13 CRAWF138 KV  TR58R04",
    "WARDAV  138 KV  COLONIAL": "WARDAV  138 KV  COLONILD",
    "PANTHER 69 KV   CRYPTOLD": "PANTHER 69 KV   CRYPT_LD",
    "SCRUBGRS13 KV   DATACEN1": "SCRUBGRS13 KV   DATCENT1",
    ############## from DAI
    "135 ELMH138 KV  TR76_12": "135 ELMH138 KV  TR76_34",
    "SHAWVILL34 KV   14 TX": "SHAWVILL34 KV   CLR2",
    "ENGLISHT35 KV   BK 5": "ENGLISHT34.5 KV A209",
    "ENGLISHT35 KV   BK 2": "ENGLISHT34.5 KV D82",
    "ENGLISHT35 KV   BK 1": "ENGLISHT34.5 KV G111",
    "SMITHBUR35 KV   LOAD1": "SMITHBUR34.5 KV TR3",
    "WIND JC 35 KV   BANK1": "WIND JC 34.5 KV G215_LD",
    "WIND JC 35 KV   BANK3": "WIND JC 34.5 KV H60",
    "WYCKOFF 115 KV  BANK3": "WYCKOFF 34.5 KV D82_LD",
    "CAPITALA138 KV  T2": "CAPITALA12 KV   T2",
    "BROSVILL138 KV  CITYDANV": "BROSVILL69 KV   CITYDANV",
    "POKAGON 138 KV  T2": "POKAGON 138 KV  T4"
}

pnode_to_bus["nodename"] = pnode_to_bus["nodename"].replace(nodename_mapping)
#############################################################

pnode_to_bus["clean_nodename"] = (
    pnode_to_bus["nodename"]
    .str.replace(r"[ _-]", "", regex=True)
    .str.upper()
    .str.replace(r"(LD|LOAD)(\d*)$", r"\2", regex=True)  # remove suffix
)


pnode_to_bus.head()

,psse_bus_#,substation,nodename,nodekey,externalnodeid,zonename,lon,lat,clean_nodename
0,10597,FOXBRKLN,FOXBRKLN230 KV LD_Y64,327047,2156118845,DOM DOMW,-78.047680,38.105092,FOXBRKLN230KVLDY64
1,12280,BLUEBELL,BLUEBELL69 KV KNOX2,327019,2156118844,FE FEOE,-81.083271,40.900624,BLUEBELL69KVKNOX2
2,2823,WOAKWOOD,WOAKWOOD69 KV T1,327166,2156118838,AEP AEPO,-84.338779,40.664493,WOAKWOOD69KVT1
3,1400,MUDDLAKE,MUDDLAKE34.5 KV AUX,327100,2156118830,AEP AEPI,-86.045797,41.956634,MUDDLAKE34.5KVAUX
4,9974,DAVESSTR,DAVESSTR34.5 KV LD3,327040,2156118822,DOM DOMN,-77.559931,38.855785,DAVESSTR34.5KV3


In [9]:
pnode_to_bus[pnode_to_bus["psse_bus_#"] == "13733"].head()

,psse_bus_#,substation,nodename,nodekey,externalnodeid,zonename,lon,lat,clean_nodename
5766,13733,CAMPGRD,CAMPGRD 69 KV LD_181,30910,1123175331,LGE LGEE,-84.120722,36.985499,CAMPGRD69KV181


In [10]:
cols = ["psse_bus_#", "substation", "nodename", "externalnodeid", "clean_nodename"]

pnode_map = pnode_to_bus[cols].copy()

hourly_load2 = hourly_load.copy()
hourly_load2["externalnodeid"] = hourly_load2["externalnodeid"].astype("Int64")
pnode_map["externalnodeid"] = pnode_map["externalnodeid"].astype("Int64")

# Optional but recommended: prevent duplicate name matches
pnode_map_by_name = pnode_map.drop_duplicates("clean_nodename", keep="first")

# 1) Match by externalnodeid
mapped = hourly_load2.merge(
    pnode_map,
    on="externalnodeid",
    how="left",
    suffixes=("", "_bus")
)

miss_ext = mapped["psse_bus_#"].isna()

# 2) Match only missed rows by clean_nodename, preserving original row index
name_matches = (
    mapped.loc[miss_ext, ["clean_nodename"]]
    .reset_index(names="_row")
    .merge(
        pnode_map_by_name,
        on="clean_nodename",
        how="left",
        suffixes=("", "_bus")
    )
    .set_index("_row")
)

# Fill missed rows by index alignment
for col in cols:
    mapped.loc[name_matches.index, col] = (
        mapped.loc[name_matches.index, col]
        .combine_first(name_matches[col])
    )
    
hourly_load_mapped = mapped

unmapped_hourly_load = hourly_load_mapped[
    hourly_load_mapped["psse_bus_#"].isna()
].copy()

In [11]:
mapped_hourly_load = hourly_load_mapped[
    hourly_load_mapped["psse_bus_#"].notna()
].copy()

hourly_load_bus_node = mapped_hourly_load[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw", "psse_bus_#", "substation"]].copy()


In [12]:
hourly_load_bus_node.info()

<class 'pandas.DataFrame'>
Index: 11593075 entries, 0 to 11648413
Data columns (total 9 columns):
 #   Column               Dtype         
---  ------               -----         
 0   datetime_ending_ept  datetime64[us]
 1   agg_nodename         string        
 2   nodename             string        
 3   distribution_factor  float64       
 4   forecast_area        string        
 5   forecast_load_mw     float64       
 6   distributed_load_mw  float64       
 7   psse_bus_#           string        
 8   substation           string        
dtypes: datetime64[us](1), float64(3), string(5)
memory usage: 1.3 GB


In [13]:
missing_nodenames = (
    hourly_load[["externalnodeid"]]
    .drop_duplicates()
    .merge(
        pnode_to_bus[["externalnodeid"]].drop_duplicates(),
        on="externalnodeid",
        how="left",
        indicator=True
    )
    .query("_merge == 'left_only'")
    .drop(columns="_merge")
    .sort_values("externalnodeid")
)

missing_nodenames


,externalnodeid
7785,205519
2763,2156118952
7601,2156118954
1678,2156118955
1641,2156118956
...,...
9786,44460745
8584,48728
3679,48896
3680,48897


In [14]:
# drop unwanted column
df = hourly_load_bus_node.drop(columns=["forecast_area"]).copy()

# reorder columns
df = df[
    [
        "datetime_ending_ept",
        "psse_bus_#",
        "substation",
        "nodename",
        "distribution_factor",
        "distributed_load_mw",
        "agg_nodename",
        "forecast_load_mw"
    ]
]

# sort values
df = df.sort_values(
    by=[
        "datetime_ending_ept",   # primary
        "psse_bus_#",            # ascending
        "nodename"               # alphabetical
    ],
    ascending=[True, True, True]
).reset_index(drop=True)

df.head()

,datetime_ending_ept,psse_bus_#,substation,nodename,distribution_factor,distributed_load_mw,agg_nodename,forecast_load_mw
0,2026-04-01,10,BERLIN,BERLIN 69 KV IBUS,0.00921,8.49531,AECO,922.0
1,2026-04-01,100,ONTARIO,ONTARIO 69 KV LOAD1,0.01120,10.32271,AECO,922.0
2,2026-04-01,100,ONTARIO,ONTARIO 69 KV LOAD2,0.01407,12.97162,AECO,922.0
3,2026-04-01,100,ONTARIO,ONTARIO 69 KV LOAD3,0.00971,8.95262,AECO,922.0
4,2026-04-01,100,ONTARIO,ONTARIO 69 KV LOAD4,0.00961,8.86134,AECO,922.0


In [15]:
# rename
df = df.rename(columns={"psse_bus_#": "BusNum"})

# IDs: X1-X9, then XA-XZ
id_pool = [f"X{i}" for i in range(1, 10)] + [f"X{c}" for c in string.ascii_uppercase]

# assign ID within each BusNum by nodename order
node_id_map = (
    df[["BusNum", "nodename"]]
    .drop_duplicates()
    .sort_values(["BusNum", "nodename"])
    .assign(
        ID=lambda x: x.groupby("BusNum").cumcount().map(lambda i: id_pool[i])
    )
)

create_aux = df.merge(node_id_map, on=["BusNum", "nodename"], how="left")
create_aux.head()

,datetime_ending_ept,BusNum,substation,nodename,distribution_factor,distributed_load_mw,agg_nodename,forecast_load_mw,ID
0,2026-04-01,10,BERLIN,BERLIN 69 KV IBUS,0.00921,8.49531,AECO,922.0,X1
1,2026-04-01,100,ONTARIO,ONTARIO 69 KV LOAD1,0.01120,10.32271,AECO,922.0,X1
2,2026-04-01,100,ONTARIO,ONTARIO 69 KV LOAD2,0.01407,12.97162,AECO,922.0,X2
3,2026-04-01,100,ONTARIO,ONTARIO 69 KV LOAD3,0.00971,8.95262,AECO,922.0,X3
4,2026-04-01,100,ONTARIO,ONTARIO 69 KV LOAD4,0.00961,8.86134,AECO,922.0,X4


## load forecast

In [16]:
load_forecast = pd.read_csv(f'data/load_frcstd_7_day/load_frcstd_7_day_{today_str}.csv', skiprows=[1])

load_forecast['forecast_load_mw'] = pd.to_numeric(load_forecast['forecast_load_mw'], errors='coerce')

load_forecast = load_forecast[['evaluated_at_datetime_ept', 'forecast_datetime_ending_ept', 'forecast_area', 'forecast_load_mw']].copy()

load_forecast["forecast_datetime_ending_ept"] = pd.to_datetime(load_forecast["forecast_datetime_ending_ept"])

forecast_to_agg = {
    "AE/MIDATL": "AECO",
    "AEP": "AEP",
    "AP": "APS",
    "ATSI": "ATSI",
    "BG&E/MIDATL": "BGE",
    "COMED": "COMED",
    "DAYTON": "DAY",
    "DEOK": "DEOK",
    "DOMINION": "DOM",
    "DP&L/MIDATL": "DPL",
    "DUQUESNE": "DUQ",
    "EKPC": "EKPC",
    "JCP&L/MIDATL": "JCPL",
    "METED/MIDATL": "METED",
    "PECO/MIDATL": "PECO",
    "PENELEC/MIDATL": "PENELEC",
    "PEPCO/MIDATL": "PEPCO",
    "PPL/MIDATL": "PPL",
    "PSE&G/MIDATL": "PSEG",
    "RECO/MIDATL": "RECO",
    "UGI/MIDATL": "UGI",
    
    # Regions
    "WESTERN_REGION": "WEST_REGION",
    "SOUTHERN_REGION": "SOUTH_REGION",
    "MID_ATLANTIC_REGION": "MIDATL_REGION",

    # Whole system
    "RTO_COMBINED": "RTO"
}

load_forecast["agg_nodename"] = load_forecast["forecast_area"].map(forecast_to_agg)

load_forecast = (
    load_forecast
    .sort_values(["forecast_datetime_ending_ept", "agg_nodename", "evaluated_at_datetime_ept"])
    .drop_duplicates(subset=["forecast_datetime_ending_ept", "agg_nodename"], keep="last")
    .reset_index(drop=True)
)

zone_load_forecast = load_forecast[load_forecast['agg_nodename'].isin(['AECO', 'AEP', 'APS', 'ATSI', 'BGE', 'COMED', 'DAY', 'DEOK', 'DOM', 'DPL', 'DUQ', 'EKPC', 'JCPL', 'METED', 'PECO', 'PENELEC', 'PEPCO', 'PPL', 'PSEG', 'RECO', 'UGI'])].copy()


In [17]:
tmr_HE12 = (datetime.today() + timedelta(days=1)).replace(hour=12, minute=0, second=0, microsecond=0)
tmr_HE18 = (datetime.today() + timedelta(days=1)).replace(hour=18, minute=0, second=0, microsecond=0)

zone_load_forecast[(zone_load_forecast["forecast_datetime_ending_ept"] == tmr_HE12) & (zone_load_forecast["agg_nodename"] == "UGI")]

,evaluated_at_datetime_ept,forecast_datetime_ending_ept,forecast_area,forecast_load_mw,agg_nodename
740539,2026-05-17 07:18:12.000,2026-05-19 12:00:00,UGI/MIDATL,122.0,UGI


In [18]:
ugi_mw = (
    zone_load_forecast.loc[zone_load_forecast["agg_nodename"] == "UGI", ["forecast_datetime_ending_ept", "forecast_load_mw"]]
    .rename(columns={"forecast_load_mw": "ugi_forecast_load_mw"})
)

zone_load_forecast = zone_load_forecast.merge(ugi_mw, on="forecast_datetime_ending_ept", how="left")

zone_load_forecast.loc[zone_load_forecast["agg_nodename"] == "PL", "forecast_load_mw"] = (
    zone_load_forecast.loc[zone_load_forecast["agg_nodename"] == "PL", "forecast_load_mw"] +
    zone_load_forecast.loc[zone_load_forecast["agg_nodename"] == "PL", "ugi_forecast_load_mw"].fillna(0)
)

zone_load_forecast = (
    zone_load_forecast.loc[zone_load_forecast["agg_nodename"] != "UGI"]
    .drop(columns="ugi_forecast_load_mw")
    .reset_index(drop=True)
)

zone_load_forecast = zone_load_forecast[["forecast_datetime_ending_ept", "agg_nodename", "forecast_load_mw"]].copy()

zone_load_forecast.rename(columns={"forecast_datetime_ending_ept": "datetime_ending_ept"}, inplace=True)

In [19]:
zone_load_forecast.head()

,datetime_ending_ept,agg_nodename,forecast_load_mw
0,2023-01-01 01:00:00,AECO,934.0
1,2023-01-01 01:00:00,AEP,12527.0
2,2023-01-01 01:00:00,APS,4772.0
3,2023-01-01 01:00:00,ATSI,6334.0
4,2023-01-01 01:00:00,BGE,2687.0


In [20]:
zone_load_forecast[zone_load_forecast['datetime_ending_ept'] == tmr_HE12]["forecast_load_mw"].sum()

np.float64(116182.0)

In [21]:
RTO_load_forecast = load_forecast[load_forecast['agg_nodename'] == 'RTO'].copy()
RTO_load_forecast[RTO_load_forecast['forecast_datetime_ending_ept'] == tmr_HE12].head()

,evaluated_at_datetime_ept,forecast_datetime_ending_ept,forecast_area,forecast_load_mw,agg_nodename
740537,2026-05-17 07:18:12.000,2026-05-19 12:00:00,RTO_COMBINED,115540.0,RTO


In [22]:

# forecast rows that need to be appended
last_dt = create_aux["datetime_ending_ept"].max()

future_zone = zone_load_forecast[
    zone_load_forecast["datetime_ending_ept"] > last_dt
].copy()

# historical distribution factors
hist_factor = create_aux[
    [
        "datetime_ending_ept",
        "BusNum",
        "substation",
        "nodename",
        "distribution_factor",
        "agg_nodename",
        "ID"
    ]
].copy()

# create forward-looking target timestamps:
# a historical factor at t-1d, t-2d, ..., t-7d can be used for target time t
factor_candidates = []

for d in range(1, 8):
    tmp = hist_factor.copy()
    tmp["datetime_ending_ept"] = tmp["datetime_ending_ept"] + pd.Timedelta(days=d)
    tmp["lag_day"] = d
    factor_candidates.append(tmp)

    # add one extra copy for yesterday and same day last week
    if d in [1, 7, 14]:
        tmp_extra = hist_factor.copy()
        tmp_extra["datetime_ending_ept"] = tmp_extra["datetime_ending_ept"] + pd.Timedelta(days=d)
        tmp_extra["lag_day"] = d
        factor_candidates.append(tmp_extra)

factor_candidates = pd.concat(factor_candidates, ignore_index=True)

# median factor across past 7 same-hour observations
median_factor = (
    factor_candidates
    .groupby(
        ["datetime_ending_ept", "BusNum", "substation", "nodename", "agg_nodename", "ID"],
        as_index=False
    )["distribution_factor"]
    .median()
)

# join median distribution factor onto forecast
append_df = future_zone.merge(
    median_factor,
    on=["datetime_ending_ept", "agg_nodename"],
    how="left"
)

# calculate bus-level distributed load
append_df["distributed_load_mw"] = (
    append_df["forecast_load_mw"] * append_df["distribution_factor"]
)

# align columns
append_df = append_df[
    [
        "datetime_ending_ept",
        "BusNum",
        "substation",
        "nodename",
        "distribution_factor",
        "distributed_load_mw",
        "agg_nodename",
        "forecast_load_mw",
        "ID"
    ]
]

# append to original table
hourly_load_forecast = pd.concat(
    [create_aux, append_df],
    ignore_index=True
)



In [23]:
hourly_load_forecast['distributed_load_mw'] = hourly_load_forecast['distributed_load_mw'].round(5)

In [24]:
hourly_load_forecast["BusNum"] = pd.to_numeric(
    hourly_load_forecast["BusNum"],
    errors="coerce"
)

hourly_load_forecast.sort_values(
    by=["datetime_ending_ept", "BusNum"],
    ascending=[True, True],
    inplace=True
)

hourly_load_forecast.head()

,datetime_ending_ept,BusNum,substation,nodename,distribution_factor,distributed_load_mw,agg_nodename,forecast_load_mw,ID
6513,2026-04-01,4,ABSECON,ABSECON 69 KV LOAD2,0.00802,7.39905,AECO,922.0,X1
8377,2026-04-01,7,ATCO,ATCO 69 KV BUS1,0.01060,9.77412,AECO,922.0,X1
8772,2026-04-01,8,BARNEGAT,BARNEGAT69 KV T1,0.00912,8.40403,AECO,922.0,X1
9475,2026-04-01,9,BECKETT,BECKETT 69 KV T1,0.01952,17.99560,AECO,922.0,X1
9476,2026-04-01,9,BECKETT,BECKETT 69 KV T2,0.01050,9.68284,AECO,922.0,X2


In [25]:
hourly_load_forecast.tail()

,datetime_ending_ept,BusNum,substation,nodename,distribution_factor,distributed_load_mw,agg_nodename,forecast_load_mw,ID
12814347,2026-05-24,21992,YARDVILL,YARDVILL13 KV T1LD,0.002125,8.38738,PSEG,3947.0,X1
12813308,2026-05-24,22014,SMECOMOR,SMECOMOR69 KV 6713,0.001920,4.77504,PEPCO,2487.0,X1
12813840,2026-05-24,22038,HUNTSVIL,HUNTSVIL13 KV T1,0.001130,3.95387,PPL,3499.0,X1
12805338,2026-05-24,22131,EWHITLEY,EWHITLEY34 KV LOAD1,0.004350,60.63465,AEP,13939.0,X1
12805339,2026-05-24,22132,EWHITLEY,EWHITLEY34 KV LOAD2,0.004350,60.63465,AEP,13939.0,X1


In [26]:
check = (
    hourly_load_forecast[
        hourly_load_forecast["datetime_ending_ept"] == tmr_HE12
    ]
    .groupby("agg_nodename")["distribution_factor"]
    .sum()
    .reset_index(name="distribution_factor_sum")
)

check["is_equal_to_1"] = np.isclose(
    check["distribution_factor_sum"], 1.0
)

check

,agg_nodename,distribution_factor_sum,is_equal_to_1
0,AECO,1.001720,False
1,AEP,0.996550,False
2,APS,1.009150,False
3,ATSI,1.008225,False
4,BGE,1.013475,False
5,COMED,1.003680,False
6,DAY,0.995230,False
7,DEOK,1.010795,False
8,DOM,0.970440,False
9,DPL,1.001660,False


In [27]:
hourly_load_forecast[hourly_load_forecast['datetime_ending_ept'] == tmr_HE12].head()

,datetime_ending_ept,BusNum,substation,nodename,distribution_factor,distributed_load_mw,agg_nodename,forecast_load_mw,ID
11705251,2026-05-19 12:00:00,4,ABSECON,ABSECON 69 KV LOAD2,0.00359,4.23979,AECO,1181.0,X1
11705279,2026-05-19 12:00:00,7,ATCO,ATCO 69 KV BUS1,0.00246,2.90526,AECO,1181.0,X1
11705280,2026-05-19 12:00:00,7,ATCO,ATCO 69 KV BUS4,0.00590,6.96790,AECO,1181.0,X2
11705289,2026-05-19 12:00:00,8,BARNEGAT,BARNEGAT69 KV T1,0.00577,6.81437,AECO,1181.0,X1
11705296,2026-05-19 12:00:00,9,BECKETT,BECKETT 69 KV T1,0.02845,33.59945,AECO,1181.0,X1


In [28]:
hourly_load_forecast[hourly_load_forecast['datetime_ending_ept'] == tmr_HE12].shape

(10200, 9)

In [29]:
hourly_load_forecast[hourly_load_forecast['datetime_ending_ept'] == tmr_HE12]['distributed_load_mw'].sum()

np.float64(115585.94074)

In [ ]:
today_str = datetime.today().strftime("%Y%m%d")

create_aux_updated = pd.read_csv(
    f"data/PWD/load_create/load_create_{today_str}.csv", skiprows=1
)

In [ ]:
create_aux_updated.head()

,LabelAppend,BusNum,ID,SMvar,SMW,Status
0,NaN,4,X1,0.0,0.0,Closed
1,NaN,7,X1,0.0,0.0,Closed
2,NaN,8,X1,0.0,0.0,Closed
3,NaN,9,X1,0.0,0.0,Closed
4,NaN,9,X2,0.0,0.0,Closed


In [ ]:
create_aux_updated.shape

(18665, 6)

In [74]:
# define tomorrow
tomorrow = pd.Timestamp.today().normalize() + pd.Timedelta(days=1)
date_str = tomorrow.strftime("%Y%m%d")

target_12 = tomorrow + pd.Timedelta(hours=12)
target_18 = tomorrow + pd.Timedelta(hours=18)

def make_load_aux(df, out_path):
    create_aux = (
        df[["BusNum", "ID", "distributed_load_mw"]]
        .drop_duplicates()
        .copy()
    )

    load_aux = create_aux_updated.merge(
        create_aux[['BusNum', 'ID', 'distributed_load_mw']],
        on=['BusNum', 'ID'],
        how='left'
    )

    # Update SMW where a match exists
    load_aux['SMW'] = load_aux['distributed_load_mw'].combine_first(
        load_aux['SMW']
    )

    # Optional: remove helper column
    load_aux.drop(columns=['distributed_load_mw'], inplace=True)

    load_aux = load_aux[["BusNum", "ID", "SMW"]].copy()

    lines = (
        load_aux["BusNum"].astype(int).astype(str)
        + ' "'
        + load_aux["ID"].astype(str)
        + '" '
        + " "
        + load_aux["SMW"].astype(str)
    )

    aux_text = (
        "Load (BusNum,ID,SMW)\n"
        "{\n"
        + "\n".join(lines)
        + "\n}"
    )

    with open(out_path, "w", encoding="utf-8") as f:
        f.write(aux_text)

    return load_aux

# # Targets
# target_sum_12 = 77000
# target_sum_18 = 80000


# 12:00 rows
df_12 = hourly_load_forecast[
    hourly_load_forecast["datetime_ending_ept"] == target_12
].copy()

# current_sum_12 = df_12["distributed_load_mw"].sum()
# scale_12 = target_sum_12 / current_sum_12

# print(current_sum_12, scale_12)

# df_12["distributed_load_mw"] = df_12["distributed_load_mw"] * scale_12

# 18:00 rows
df_18 = hourly_load_forecast[
    hourly_load_forecast["datetime_ending_ept"] == target_18
].copy()

# current_sum_18 = df_18["distributed_load_mw"].sum()
# scale_18 = target_sum_18 / current_sum_18

# print(current_sum_18, scale_18)

# df_18["distributed_load_mw"] = df_18["distributed_load_mw"] * scale_18

# dynamic paths
path_12 = f"data/PWD/load_HE/load_{date_str}_HE12.aux"
path_18 = f"data/PWD/load_HE/load_{date_str}_HE18.aux"



In [75]:
# create files
create_aux_12 = make_load_aux(df_12, path_12)
create_aux_18 = make_load_aux(df_18, path_18)

In [76]:
def make_load_csv(create_aux, out_path):
    with open(out_path, "w", encoding="utf-8", newline="") as f:
        f.write("Load\n")
        create_aux.to_csv(
            f,
            index=False,
            lineterminator="\n"
        )


csv_path_12 = f"data/PWD/load_HE/load_{date_str}_HE12.csv"
csv_path_18 = f"data/PWD/load_HE/load_{date_str}_HE18.csv"

# create CSV files
make_load_csv(create_aux_12, csv_path_12)
make_load_csv(create_aux_18, csv_path_18)

# output hourly load by distribution

In [32]:
hourly_load_forecast.tail()

,datetime_ending_ept,BusNum,substation,nodename,distribution_factor,distributed_load_mw,agg_nodename,forecast_load_mw,ID
12814347,2026-05-24,21992,YARDVILL,YARDVILL13 KV T1LD,0.002125,8.38738,PSEG,3947.0,X1
12813308,2026-05-24,22014,SMECOMOR,SMECOMOR69 KV 6713,0.001920,4.77504,PEPCO,2487.0,X1
12813840,2026-05-24,22038,HUNTSVIL,HUNTSVIL13 KV T1,0.001130,3.95387,PPL,3499.0,X1
12805338,2026-05-24,22131,EWHITLEY,EWHITLEY34 KV LOAD1,0.004350,60.63465,AEP,13939.0,X1
12805339,2026-05-24,22132,EWHITLEY,EWHITLEY34 KV LOAD2,0.004350,60.63465,AEP,13939.0,X1


In [33]:
dtype_map = {
    "agg_pjm_ExternalNodeID": "string",
    "pjm_ExternalNodeID": "string",
    "agg_NodeName": "string",
    "NodeName": "string",
    "forecast_area": "string",
    "is_valid": "int8"
}

offset = 0
today_str = (datetime.today() - timedelta(days=offset)).strftime("%m%d") 

hourly_load = pd.read_csv(
    f"data/hourly_load_MW_by_distribution/hourly_load_MW_by_distribution_{today_str}.csv",
    skiprows=[1],
    dtype=dtype_map,
    parse_dates=["datetime_ending_ept", "insert_datetime"]
)

hourly_load.tail()

,autokey,datetime_ending_ept,agg_pjm_ExternalNodeID,agg_NodeName,pjm_ExternalNodeID,NodeName,distribution_factor,forecast_area,forecast_load_mw,distributed_load_mw,is_valid,insert_datetime
11698565,101349490,2026-05-18 04:00:00,51299,PPL,2041945254,LYCOMING69 KV HEPB2_LD,0.00513,PPL/MIDATL,3597.0,18.43822,1,2026-05-18 08:40:36
11698566,101349491,2026-05-18 04:00:00,51299,PPL,2041945256,LYCOMING69 KV HEPB1_LD,0.00235,PPL/MIDATL,3597.0,8.44935,1,2026-05-18 08:40:36
11698567,101349492,2026-05-18 04:00:00,51299,PPL,2156108938,LOOMIS 13 KV LOAD2,0.00078,PPL/MIDATL,3597.0,2.79847,1,2026-05-18 08:40:36
11698568,101349493,2026-05-18 04:00:00,51299,PPL,2156108939,LOOMIS 13 KV LOAD1,0.00078,PPL/MIDATL,3597.0,2.79847,1,2026-05-18 08:40:36
11698569,101349494,2026-05-18 04:00:00,51299,PPL,2156112972,LONGPOND69 KV LOPO_LD,0.00162,PPL/MIDATL,3597.0,5.81635,1,2026-05-18 08:40:36


In [34]:
# Ensure datetime type
hourly_load_forecast["datetime_ending_ept"] = pd.to_datetime(
    hourly_load_forecast["datetime_ending_ept"]
)

hourly_load["datetime_ending_ept"] = pd.to_datetime(
    hourly_load["datetime_ending_ept"]
)

# Tomorrow filter
today = pd.Timestamp.today().normalize()
tomorrow = today + pd.Timedelta(days=1)
day_after = tomorrow + pd.Timedelta(days=1)

forecast_df = hourly_load_forecast[
    (hourly_load_forecast["datetime_ending_ept"] >= tomorrow) &
    (hourly_load_forecast["datetime_ending_ept"] < day_after)
].copy()

# Build mapping from hourly_load using NodeName
# (drop duplicates just in case)
node_mapping = (
    hourly_load[
        [
            "NodeName",
            "agg_pjm_ExternalNodeID",
            "pjm_ExternalNodeID"
        ]
    ]
    .dropna(subset=["NodeName"])
    .drop_duplicates(subset=["NodeName"])
)

# Merge mapping into forecast data
forecast_df = forecast_df.merge(
    node_mapping,
    left_on="nodename",
    right_on="NodeName",
    how="left"
)

# Build final dataframe in hourly_load schema
hourly_load_fcst = pd.DataFrame({
    "autokey": "",
    "datetime_ending_ept": forecast_df["datetime_ending_ept"],
    "agg_pjm_ExternalNodeID": forecast_df["agg_pjm_ExternalNodeID"],
    "agg_NodeName": forecast_df["agg_nodename"],
    "pjm_ExternalNodeID": forecast_df["pjm_ExternalNodeID"],
    "NodeName": forecast_df["nodename"],
    "distribution_factor": forecast_df["distribution_factor"],
    "forecast_area": forecast_df["agg_nodename"],
    "forecast_load_mw": forecast_df["forecast_load_mw"],
    "distributed_load_mw": forecast_df["distributed_load_mw"],
    "is_valid": 1,
    "insert_datetime": ""
})

# Optional: enforce same column order
hourly_load_fcst = hourly_load_fcst[hourly_load.columns]

# drop columns
hourly_load_fcst = hourly_load_fcst.drop(columns=["autokey", "insert_datetime"])

hourly_load_fcst.head()

,datetime_ending_ept,agg_pjm_ExternalNodeID,agg_NodeName,pjm_ExternalNodeID,NodeName,distribution_factor,forecast_area,forecast_load_mw,distributed_load_mw,is_valid
0,2026-05-19,51291,AECO,32947101,ABSECON 69 KV LOAD2,0.00797,AECO,1203.0,9.58550,1
1,2026-05-19,51291,AECO,49786,ATCO 69 KV BUS1,0.00685,AECO,1203.0,8.23574,1
2,2026-05-19,51291,AECO,49787,ATCO 69 KV BUS4,0.00842,AECO,1203.0,10.12565,1
3,2026-05-19,51291,AECO,74008619,BARNEGAT69 KV T1,0.00864,AECO,1203.0,10.39512,1
4,2026-05-19,51291,AECO,2156111257,BECKETT 69 KV T1,0.02368,AECO,1203.0,28.48584,1


In [35]:
hourly_load_fcst.head()

,datetime_ending_ept,agg_pjm_ExternalNodeID,agg_NodeName,pjm_ExternalNodeID,NodeName,distribution_factor,forecast_area,forecast_load_mw,distributed_load_mw,is_valid
0,2026-05-19,51291,AECO,32947101,ABSECON 69 KV LOAD2,0.00797,AECO,1203.0,9.58550,1
1,2026-05-19,51291,AECO,49786,ATCO 69 KV BUS1,0.00685,AECO,1203.0,8.23574,1
2,2026-05-19,51291,AECO,49787,ATCO 69 KV BUS4,0.00842,AECO,1203.0,10.12565,1
3,2026-05-19,51291,AECO,74008619,BARNEGAT69 KV T1,0.00864,AECO,1203.0,10.39512,1
4,2026-05-19,51291,AECO,2156111257,BECKETT 69 KV T1,0.02368,AECO,1203.0,28.48584,1


In [ ]:

# tmr_str = tomorrow.strftime("%y%m%d")

# hourly_load_fcst.to_csv(f"data/hourly_load_MW_by_distribution_factors_frcst/hourly_load_MW_by_distribution_factors_frcst_{tmr_str}.csv", index=False)

# hourly_load_fcst.to_excel(f"data/hourly_load_MW_by_distribution_factors_frcst/hourly_load_MW_by_distribution_factors_frcst_{tmr_str}.xlsx", engine = "openpyxl", index=False)

In [39]:
import pyodbc

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 13 for SQL Server};"
    "SERVER=10.1.10.243;"
    "DATABASE=QA;"
    "UID=cshi;"
    "PWD=Dkh$2910;"
    "TrustServerCertificate=yes;"
)

cursor = conn.cursor()
cursor.fast_executemany = True

upload_df = hourly_load_fcst.copy()

upload_df = upload_df[
    [
        "datetime_ending_ept",
        "agg_pjm_ExternalNodeID",
        "agg_NodeName",
        "pjm_ExternalNodeID",
        "NodeName",
        "distribution_factor",
        "forecast_area",
        "forecast_load_mw",
        "distributed_load_mw",
        "is_valid",
    ]
]

upload_df["insert_datetime"] = pd.Timestamp.now()

# Get current max autokey from SQL table
cursor.execute("""
    SELECT ISNULL(MAX([autokey]), 0)
    FROM [QA].[dbo].[hourly_load_MW_by_distribution_factors_frcst]
""")

max_autokey = cursor.fetchone()[0]

# Add autokey as first column
upload_df.insert(
    0,
    "autokey",
    range(max_autokey + 1, max_autokey + 1 + len(upload_df))
)

# Convert pandas/numpy values to native Python values
def to_python_value(x):
    if pd.isna(x):
        return None
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, np.floating):
        return float(x)
    if isinstance(x, np.bool_):
        return bool(x)
    if isinstance(x, pd.Timestamp):
        return x.to_pydatetime()
    return x

data = [
    tuple(to_python_value(x) for x in row)
    for row in upload_df.itertuples(index=False, name=None)
]

sql = """
INSERT INTO [QA].[dbo].[hourly_load_MW_by_distribution_factors_frcst] (
    [autokey],
    [datetime_ending_ept],
    [agg_pjm_ExternalNodeID],
    [agg_NodeName],
    [pjm_ExternalNodeID],
    [NodeName],
    [distribution_factor],
    [forecast_area],
    [forecast_load_mw],
    [distributed_load_mw],
    [is_valid],
    [insert_datetime]
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

try:
    cursor.executemany(sql, data)
    conn.commit()
    print(f"Uploaded {len(upload_df):,} rows.")

except pyodbc.Error as e:
    conn.rollback()
    print("Upload failed:")
    print(e)

finally:
    cursor.close()
    conn.close()

Uploaded 244,489 rows.


In [96]:
upload_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 244721 entries, 0 to 244720
Data columns (total 11 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   datetime_ending_ept     244721 non-null  datetime64[us]
 1   agg_pjm_ExternalNodeID  244721 non-null  string        
 2   agg_NodeName            244721 non-null  object        
 3   pjm_ExternalNodeID      244721 non-null  string        
 4   NodeName                244721 non-null  string        
 5   distribution_factor     244721 non-null  float64       
 6   forecast_area           244721 non-null  object        
 7   forecast_load_mw        244721 non-null  float64       
 8   distributed_load_mw     244721 non-null  float64       
 9   is_valid                244721 non-null  int64         
 10  insert_datetime         244721 non-null  datetime64[us]
dtypes: datetime64[us](2), float64(3), int64(1), object(2), string(3)
memory usage: 29.0+ MB


# check only

In [ ]:
base_case = pd.read_csv('data/PWD/base case.csv', skiprows = [0])
base_case = base_case[["Label", "BusNum", "AreaName", "LoadID", "LoadMW",  "LoadStatus"]]
base_case.rename(columns={"Label": "LabelAppend", "LoadID": "ID", "LoadMW": "LoadSMW", "LoadStatus": "Status"}, inplace=True)
base_case["ID"] = base_case["ID"].astype(str)
base_case = base_case[base_case["AreaName"] != "PJM"].copy()
base_case.head()

,LabelAppend,BusNum,AreaName,ID,LoadSMW,Status
0,159-1,159,AECI,1,0.0,Closed
1,203-1,203,AECI,1,0.0,Closed
2,242-3,242,AECI,3,0.0,Closed
3,172-1,172,AECI,1,0.0,Closed
4,247-2,247,AECI,2,0.0,Closed


In [123]:
base_case[base_case["AreaName"] == "PJM"].head()

,LabelAppend,BusNum,AreaName,ID,LoadSMW,Status
5676,4322-2,4322,PJM,2,6.86,Closed
5677,4322-1,4322,PJM,1,6.86,Closed
5678,2121-1,2121,PJM,1,24.33,Closed
5679,4321-1,4321,PJM,1,10.13,Closed
5680,4319-2,4319,PJM,2,1.33,Closed


In [118]:
base_case.shape

(19908, 6)

In [128]:
load_forecast_HE12 = hourly_load_forecast[hourly_load_forecast['datetime_ending_ept'] == tmr_HE12].copy()
load_forecast_HE18 = hourly_load_forecast[hourly_load_forecast['datetime_ending_ept'] == tmr_HE18].copy()

load_forecast_HE12.head()

,datetime_ending_ept,BusNum,substation,nodename,distribution_factor,distributed_load_mw,agg_nodename,forecast_load_mw,ID
10256359,2026-05-13 12:00:00,4,ABSECON,ABSECON 69 KV LOAD2,0.00541,3.55437,AECO,657.0,X1
10256388,2026-05-13 12:00:00,7,ATCO,ATCO 69 KV BUS1,0.00472,3.10104,AECO,657.0,X1
10256389,2026-05-13 12:00:00,7,ATCO,ATCO 69 KV BUS4,0.00634,4.16538,AECO,657.0,X2
10256398,2026-05-13 12:00:00,8,BARNEGAT,BARNEGAT69 KV T1,0.00819,5.38083,AECO,657.0,X1
10256406,2026-05-13 12:00:00,9,BECKETT,BECKETT 69 KV T2,0.01172,7.70004,AECO,657.0,X2


In [131]:
import string

def map_id(val):
    # remove leading X
    suffix = str(val).replace('X', '')

    # numeric case: X1-X9 -> 1-9
    if suffix.isdigit():
        return int(suffix)

    # alphabetic case: XA-XZ -> 10-35
    if suffix.isalpha():
        return ord(suffix.upper()) - ord('A') + 10

    return None

# create mapped value
load_forecast_HE12['ID_mapped'] = load_forecast_HE12['ID'].apply(map_id)

# create LabelAppend column
load_forecast_HE12['LabelAppend'] = (
    load_forecast_HE12['BusNum'].astype(str)
    + '-'
    + load_forecast_HE12['ID_mapped'].astype(str)
)


# create mapped value
load_forecast_HE18['ID_mapped'] = load_forecast_HE18['ID'].apply(map_id)

# create LabelAppend column
load_forecast_HE18['LabelAppend'] = (
    load_forecast_HE18['BusNum'].astype(str)
    + '-'
    + load_forecast_HE18['ID_mapped'].astype(str)
)

load_forecast_HE12.head()

,datetime_ending_ept,BusNum,substation,nodename,distribution_factor,distributed_load_mw,agg_nodename,forecast_load_mw,ID,ID_mapped,LabelAppend
10256359,2026-05-13 12:00:00,4,ABSECON,ABSECON 69 KV LOAD2,0.00541,3.55437,AECO,657.0,X1,1,4-1
10256388,2026-05-13 12:00:00,7,ATCO,ATCO 69 KV BUS1,0.00472,3.10104,AECO,657.0,X1,1,7-1
10256389,2026-05-13 12:00:00,7,ATCO,ATCO 69 KV BUS4,0.00634,4.16538,AECO,657.0,X2,2,7-2
10256398,2026-05-13 12:00:00,8,BARNEGAT,BARNEGAT69 KV T1,0.00819,5.38083,AECO,657.0,X1,1,8-1
10256406,2026-05-13 12:00:00,9,BECKETT,BECKETT 69 KV T2,0.01172,7.70004,AECO,657.0,X2,2,9-2


In [133]:
load_forecast_HE12.shape

(10205, 11)

In [130]:
load_forecast_HE12_filtered = load_forecast_HE12[
    ~load_forecast_HE12["LabelAppend"].isin(base_case["LabelAppend"])
].copy()

load_forecast_HE12_filtered.head()

,datetime_ending_ept,BusNum,substation,nodename,distribution_factor,distributed_load_mw,agg_nodename,forecast_load_mw,ID,ID_mapped,LabelAppend


In [134]:
load_forecast_HE18_filtered = load_forecast_HE18[
    ~load_forecast_HE18["LabelAppend"].isin(base_case["LabelAppend"])
].copy()

load_forecast_HE18_filtered.head()

,datetime_ending_ept,BusNum,substation,nodename,distribution_factor,distributed_load_mw,agg_nodename,forecast_load_mw,ID,ID_mapped,LabelAppend


# sanity check

In [32]:
df_12['distributed_load_mw'].sum()

np.float64(82483.22662)

In [33]:
df_12.shape

(10204, 9)

In [34]:
df_12.head()

,datetime_ending_ept,BusNum,substation,nodename,distribution_factor,distributed_load_mw,agg_nodename,forecast_load_mw,ID
9744868,2026-05-12 12:00:00,4,ABSECON,ABSECON 69 KV LOAD2,0.00248,1.51032,AECO,609.0,X1
9744898,2026-05-12 12:00:00,7,ATCO,ATCO 69 KV BUS4,0.00742,4.51878,AECO,609.0,X2
9744897,2026-05-12 12:00:00,7,ATCO,ATCO 69 KV BUS1,0.00450,2.74050,AECO,609.0,X1
9744907,2026-05-12 12:00:00,8,BARNEGAT,BARNEGAT69 KV T1,0.00543,3.30687,AECO,609.0,X1
9744914,2026-05-12 12:00:00,9,BECKETT,BECKETT 69 KV T1,0.03615,22.01535,AECO,609.0,X1


In [159]:
df_12[df_12["BusNum"] == 13733].head()

,datetime_ending_ept,BusNum,substation,nodename,distribution_factor,distributed_load_mw,agg_nodename,forecast_load_mw,ID,distributed_load_mw_r2,Label
9025721,2026-05-09 12:00:00,13733,CAMPGRD,CAMPGRD 69 KV LD_181,0.00344,4.38944,EKPC,1276.0,X1,4.39,13733-X1


In [151]:
df_12["Label"] = df_12["BusNum"].astype(str) + "-" + df_12["ID"].astype(str)

In [139]:
load = pd.read_csv("data/PWD/pjm_load.csv", skiprows=[0])
load.head()

,LabelAppend,BusNum,ID,SMvar,SMW,Label,BusName,AreaName,ZoneName,Status,MW
0,NaN,4,X1,0.0,3.59,4-X1,ABSECON,PJM,AE AE,Closed,3.59
1,NaN,7,X1,0.0,3.75,7-X1,ATCO,PJM,AE AE,Closed,3.75
2,NaN,7,X2,0.0,5.36,7-X2,ATCO,PJM,AE AE,Closed,5.36
3,NaN,8,X1,0.0,5.44,8-X1,BARNEGAT,PJM,AE AE,Closed,5.44
4,NaN,9,X1,0.0,20.92,9-X1,BECKETT,PJM,AE AE,Closed,20.92


In [141]:
load = load[load["AreaName"] == "PJM"].copy()

In [143]:
load.shape

(10346, 11)

In [144]:
load.head()

,LabelAppend,BusNum,ID,SMvar,SMW,Label,BusName,AreaName,ZoneName,Status,MW
0,NaN,4,X1,0.0,3.59,4-X1,ABSECON,PJM,AE AE,Closed,3.59
1,NaN,7,X1,0.0,3.75,7-X1,ATCO,PJM,AE AE,Closed,3.75
2,NaN,7,X2,0.0,5.36,7-X2,ATCO,PJM,AE AE,Closed,5.36
3,NaN,8,X1,0.0,5.44,8-X1,BARNEGAT,PJM,AE AE,Closed,5.44
4,NaN,9,X1,0.0,20.92,9-X1,BECKETT,PJM,AE AE,Closed,20.92


In [145]:
load["MW"].sum()

np.float64(75870.17)

In [163]:
# Round distributed_load_mw to 2 decimals
df_12["distributed_load_mw_r2"] = df_12["distributed_load_mw"].round(2)

# Merge with load on BusNum
merged = df_12.merge(
    load[["Label", "MW"]],
    on="Label",
    how="left"
)

# Compute difference
merged["diff"] = merged["distributed_load_mw_r2"] - merged["MW"]

# Show rows where difference is not 0
non_zero_diff = merged[merged["diff"] != 0][
    [
        "Label",
        "distributed_load_mw",
        "distributed_load_mw_r2",
        "MW",
        "diff"
    ]
]

# filter when MW is NaN
non_zero_diff[non_zero_diff["MW"].isna()]

,Label,distributed_load_mw,distributed_load_mw_r2,MW,diff
7680,13560-X1,1.83744,1.84,NaN,NaN
7681,13562-X1,4.22356,4.22,NaN,NaN
7682,13563-X1,0.81664,0.82,NaN,NaN
7683,13564-X1,2.79444,2.79,NaN,NaN
7684,13565-X1,3.21552,3.22,NaN,NaN
7685,13565-X2,4.88708,4.89,NaN,NaN
7686,13566-X1,13.55112,13.55,NaN,NaN
7687,13567-X1,5.98444,5.98,NaN,NaN
7688,13568-X1,2.79444,2.79,NaN,NaN
7689,13569-X1,4.51704,4.52,NaN,NaN


In [115]:
import pandas as pd

# ensure ID is string
df = load.copy()
df["ID"] = df["ID"].astype(str)

# flag whether ID starts with X
df["is_X"] = df["ID"].str.startswith("X")

# for each BusNum, check if both types exist
group_has_X = df.groupby("BusNum")["is_X"].transform("any")
group_has_num = df.groupby("BusNum")["is_X"].transform(lambda x: (~x).any())

# keep logic
filtered = df[
    ~(group_has_X & group_has_num) | df["is_X"]
].drop(columns="is_X")

In [116]:
filtered.head()


,LabelAppend,BusNum,ID,SMvar,SMW,Label,BusName,AreaName,ZoneName,Status,MW
5720,NaN,2289,X1,0.0,20.01,2289-X1,LUNDY,PJM,AEP AEPO,Closed,20.01
5721,NaN,1144,1,0.0,0.00,1144-1,CREEKWLK,PJM,AEP AEPI,Closed,0.00
5723,NaN,2288,X1,0.0,12.65,2288-X1,LUMBERJK,PJM,AEP AEPO,Closed,12.65
5724,NaN,1143,X2,0.0,3.38,1143-X2,COWANIM,PJM,AEP AEPI,Closed,3.38
5727,NaN,2287,X2,0.0,1.91,2287-X2,LSII,PJM,AEP AEPO,Closed,1.91


In [117]:
filtered["MW"].sum()

np.float64(82858.55)

In [118]:
# make sure ID is string
df = filtered.copy()
df["ID"] = df["ID"].astype(str)

# boolean mask
is_X = df["ID"].str.startswith("X")

# counts
count_X = is_X.sum()
count_not_X = (~is_X).sum()

count_X, count_not_X

(np.int64(9917), np.int64(1416))

In [119]:
filtered["ID"] = filtered["ID"].astype(str)

mw_sum_X = filtered.loc[
    filtered["ID"].str.startswith("X"), "MW"
].sum()

mw_sum_X

np.float64(75002.18)

In [ ]:
not_X_sorted = df.loc[~is_X].sort_values(by="MW", ascending=False)

not_X_sorted

,LabelAppend,BusNum,ID,SMvar,SMW,Label,BusName,AreaName,ZoneName,Status,MW
22612,NaN,16682,1,0.0,216.14,16682-1,PLYMOUTH,PJM,PE PE,Closed,216.14
15889,NaN,11342,1,0.0,147.47,11342-1,BRUNOTIS,PJM,DUQ DUQU,Closed,147.47
22684,NaN,16774,2,0.0,123.37,16774-2,WMORELAN,PJM,PE PE,Closed,123.37
22683,NaN,16774,1,0.0,123.37,16774-1,WMORELAN,PJM,PE PE,Closed,123.37
22685,NaN,16774,3,0.0,123.37,16774-3,WMORELAN,PJM,PE PE,Closed,123.37
...,...,...,...,...,...,...,...,...,...,...,...
14662,NaN,10347,1,0.0,-4.97,10347-1,EVERETTS,PJM,DOM DOMS,Closed,-4.97
22613,NaN,16683,1,0.0,-4.99,16683-1,PLYMOUTH,PJM,PE PE,Closed,-4.99
10604,NaN,391,1,0.0,-6.33,391-1,CITYMART,PJM,AEP AEPA,Closed,-6.33
14899,NaN,10584,1,0.0,-13.07,10584-1,ENDCAVRN,PJM,DOM DOMW,Closed,-13.07
